# CRISP-DM · Phase 1–2 — Business Understanding & Data Understanding
## Business Objectives (BO) & Data Science Objectives (DSO)

**Project:** Cloud-Native AI Operations Agent for CEM–CVM Intelligence — *Telecom NeXoligence*  
**Author:** Souhayl Guenichi · ESPRIT engineering student · 6-month internship, Huawei Tunisia (Cloud IT / Sales-Solution)  
**Client:** Tunisie Telecom (TT) — real OSS + BSS production data, accessed via Huawei Tunisia  
**Methodology:** Hybrid **CRISP-DM** + MLOps overlay  
**Strategic frame:** Huawei **ADN** (Autonomous Driving Network) — the intelligence layer for **O+B convergence**

---

> This notebook is the formal record of the first two CRISP-DM phases. It is **documentation only** (no code, no model artifacts). It fixes *what we are trying to achieve* (Business Understanding) and *what data makes it achievable* (Data Understanding) **before** a single model is trained. Every downstream notebook (`00`→`10`) and every deployed model exists to satisfy an objective stated here.

## 0. Why these two phases come first

CRISP-DM (Cross-Industry Standard Process for Data Mining) is the de-facto lifecycle for industrial data-science projects. Its first two phases answer two distinct questions:

| Phase | Question it answers | Output of *this* notebook |
|---|---|---|
| **1. Business Understanding** | *What business problem are we solving, and how do we know we succeeded?* | 4 **Business Objectives** + success metrics |
| **2. Data Understanding** | *What data do we have, and can it support those objectives?* | Data inventory + 4 **Data Science Objectives** that translate the business goals into measurable ML targets |

**The contract that makes a defense convincing:** every *Data Science Objective* must trace back to a *Business Objective*, and every Business Objective must trace back to a real, demonstrable operator pain. We make that traceability explicit in §4.

## 1. Business Context — the pain we exist to remove

### 1.1 The operator's blind spot (defense pain hook)

> **"Network anomalies stay invisible to OSS until a customer complaint reaches Care."**

In a classical telco, two worlds run in parallel and rarely talk:

- **OSS (Operations Support Systems)** watches the *network*: cell KPIs, RAT availability, throughput, drop rates.
- **BSS / CEM (Business Support Systems / Customer Experience Management)** watches the *subscriber*: usage, device, perceived experience, churn signals.

When a cell silently degrades, OSS thresholds often stay green while real customers already suffer. The degradation only becomes "real" to the operator when **enough subscribers complain to Care** — by which point experience is damaged, churn risk is up, and the fix is reactive and expensive.

### 1.2 Where this project sits (Huawei ADN)

```
[CEM / SmartCare] ──▶ [AI Operations Agent] ──▶ [CVM]
                           ▲ this project ▲
                     (cloud-native, ADN L4)
```

The project is the **intelligence layer** of Huawei's Autonomous Driving Network vision. It reads CEM/OSS outputs, fuses them with BSS subscriber data, and feeds actionable intelligence to Customer Value Management (CVM). The four ADN pillars it demonstrates: **O+B convergence**, **CEM/SmartCare demarcation**, **agentic AI**, and a **CVM output layer**.

### 1.3 Out of scope (locked)

Billing/revenue analytics, LSTM churn model, real NOC alarm baseline, and HCS cloud migration are explicitly **out of scope** for this milestone. The orientation is **CEM subscriber experience profiling** (aligned with Huawei SmartCare), *not* billing.

## 2. Business Objectives (BO)

Four objectives, ordered by how directly they attack the §1.1 pain. Each states the goal, the business rationale, and a **business-level success metric** (an outcome the operator can feel — not an ML score).

---

### BO1 — Eliminate the OSS↔Care blind spot
**Objective.** Detect network-experience degradation **proactively**, surfacing it *before* it reaches Care as a customer complaint.

**Why it matters.** This is the core operator pain (§1.1). Every cycle of "degrade → complain → react" costs experience, loyalty, and OPEX. Moving detection upstream of the complaint converts reactive firefighting into proactive operations.

**Business success metric.** Anomalous cells flagged by the platform **lead** the corresponding rise in Care-relevant experience signals (positive detection lead-time, measured via Granger lag in DSO4); degraded experience is identifiable at the cell/area level without waiting for subscriber complaints.

---

### BO2 — Make decisions on converged O+B intelligence
**Objective.** Fuse **OSS network KPIs** with **BSS subscriber experience** at the **geographic area** level to produce a single, unified operational picture.

**Why it matters.** O+B convergence is the headline pillar of Huawei ADN. There is no direct IMSI↔cell link in the data, so **geographic area is the join key**. A converged view lets one signal (a degrading cell) be reasoned about together with its human impact (subscribers in that area).

**Business success metric.** Every governorate/area resolves to a combined OSS-health + CEM-experience profile, and statistically significant OSS→CEM relationships are surfaced (p < 0.05) rather than assumed.

---

### BO3 — Move operations toward ADN Level-4 autonomy
**Objective.** Let the platform **auto-triage and auto-remediate safe actions**, while routing risky actions to a human for approval (human-in-the-loop).

**Why it matters.** ADN L4 = the network largely runs itself. Auto-approving low-risk, high-confidence actions (info, predictions) and gating critical/warning remediations reduces **Mean-Time-To-Resolution** and **OPEX**, while keeping a human accountable for consequential changes.

**Business success metric.** Safe actions execute autonomously with a full persisted audit trail; risky actions are correctly withheld for human approval; resolution happens without manual triage of routine cases.

---

### BO4 — Protect experience and reduce churn risk
**Objective.** Identify **underserved and at-risk subscribers** (poor radio access, low experience score) and feed them to proactive retention (the CVM layer).

**Why it matters.** Underservice is a leading indicator of churn. Catching it early — *before* the subscriber decides to leave — turns a likely loss into a retention opportunity, which is the entire purpose of the downstream CVM system.

**Business success metric.** Subscribers can be ranked by experience score and underservice risk; the cohort meeting the churn-risk rule (high RAT gap **and** low CEM score) is identifiable and actionable for retention playbooks.

## 3. Data Science Objectives (DSO)

Each DSO is the **measurable ML translation** of one or more business objectives. Crucially, every DSO is backed by a **real, trained, deployed model** in this repository — these are not aspirations. Success criteria are stated as **targets** (CRISP-DM goals); achieved values live in the model cards / `metrics.json` and are reported honestly in the training notebooks.

---

### DSO1 — CEM Experience Scoring (supervised regression)
| Field | Specification |
|---|---|
| **ML task** | Regress a per-subscriber **experience score in [0, 1]** from BSS behavioural + device + network-quality features |
| **Model** | LightGBM (DART, 256 leaves, depth 12) |
| **Input data** | `subscriber_features` derived from real + simulated BSS (2.47M subscribers, 5 months) |
| **Success target** | **Test R² ≥ 0.95**, low MAE — a reliable continuous experience signal |
| **Serves** | **BO1** (per-subscriber experience visibility), **BO4** (rank by experience for retention) |

---

### DSO2 — Experience Anomaly Detection (unsupervised)
| Field | Specification |
|---|---|
| **ML task** | Flag **abnormal cell behaviour** in OSS KPIs without labels — learn "normal", score deviation |
| **Model** | PyTorch **VAE** (9→32→16→latent 8), trained **normal-only** |
| **Input data** | OSS cell KPIs (1M records, normal-only training subset of 18.8M real) |
| **Success target** | **ROC-AUC ≥ 0.90** and **Recall ≥ 0.70** — catch real degradations with few misses |
| **Serves** | **BO1** (proactive degradation detection), **BO3** (anomalies trigger autonomous triage) |

---

### DSO3 — RAT Underservice Classification (supervised classification)
| Field | Specification |
|---|---|
| **ML task** | Classify subscribers **underserved on radio-access technology** (e.g. stuck on weaker RAT) |
| **Model** | XGBoost (500 trees, depth 8, GPU) |
| **Input data** | `subscriber_features` (2.47M subscribers) — leakage-controlled feature set |
| **Success target** | **ROC-AUC ≥ 0.95** with an honest, leakage-free feature set (verified by CV) |
| **Serves** | **BO4** (identify the at-risk / underserved cohort for retention) |

---

### DSO4 — OSS→CEM Causal Lead-Time (temporal causality)
| Field | Specification |
|---|---|
| **ML task** | Determine **which OSS KPIs lead CEM degradation, and by how many cycles** — then forecast CEM from leading OSS signals |
| **Method** | **Granger causality** F-test gate (offline) → lagged OLS forecast (online); `CEM_Y(t) ~ OSS_X(t−best_lag)` |
| **Input data** | Time-aligned OSS + CEM series per area (two-tier: offline gate `granger_feature_gate.json` + online lead-time API) |
| **Success target** | Statistically **significant OSS→CEM pairs at p < 0.05** with identified lag, enabling forward projection |
| **Serves** | **BO1** (lead-time = detect before complaint), **BO2** (the quantified O+B convergence link) |

## 4. BO → DSO Traceability Matrix

This is the heart of a convincing defense: **no orphan objectives**. Every business goal is served by at least one measurable ML objective, and every ML objective earns its place by serving a business goal.

| | **DSO1** CEM Score | **DSO2** Anomaly | **DSO3** RAT Underservice | **DSO4** Granger Lead-time |
|---|:---:|:---:|:---:|:---:|
| **BO1** — Kill the OSS↔Care blind spot | ✅ | ✅ | | ✅ |
| **BO2** — Converged O+B decisioning | | | | ✅ |
| **BO3** — ADN L4 autonomy | | ✅ | | |
| **BO4** — Protect experience / cut churn | ✅ | | ✅ | |

**Reading the matrix.**
- **BO1** is the most-supported objective (3 DSOs) — correct, because it *is* the project's reason to exist.
- **BO2** is proven specifically by **DSO4**: Granger causality is what turns "OSS and BSS are correlated" into "OSS *leads* CEM by N cycles with p < 0.05" — a defensible convergence claim, not a hand-wave.
- **BO3**'s autonomy is fed by **DSO2** anomalies (the triggers the L4 agent acts on).
- **BO4** retention combines **DSO1** (low score) and **DSO3** (underservice) into the churn-risk cohort.
- **No empty column** — every model is justified. **No empty row** — every business promise is backed.

## 5. Data Understanding — can the data support these objectives?

An objective is only credible if the data exists to pursue it. Inventory of the real Tunisie Telecom data backing each DSO:

### 5.1 Data inventory

| Source | Volume | Nature | Feeds |
|---|---|---|---|
| **BSS subscribers (real)** | **968,077** (Feb 468K + Mar 500K) | 26 features/subscriber — usage, device, network-quality | DSO1, DSO3 |
| **BSS subscribers (simulated)** | **1.5M** (Jan/Apr/May) | Stratified bootstrap + log-normal perturbation to fill missing months | DSO1, DSO3 |
| **OSS cell KPIs (real)** | **18.8M** (2G 3.4M + 3G 6.9M + 4G 8.5M) | Per-cell radio/network KPIs | DSO2, DSO4 |
| **OSS cell KPIs (simulated)** | **200K** (Jan/Feb/May/Jun) | Bootstrap reservoir with temporal drift | DSO2, DSO4 |
| **Derived features** | `subscriber_features` + `area_network_health` (6 months) | Engineered ML inputs | all DSO |

> **Confidentiality.** All real TT data lives in the git-ignored `TT_data/` directory and **never leaves the host**. It is used strictly for local training and inference within this project.

### 5.2 Key data facts that shape the objectives

1. **No direct IMSI↔cell link.** BSS (subscriber) and OSS (cell) cannot be joined per-record. → **geographic area is the join key** (drives BO2 / DSO4 being area-level).
2. **Anomaly labels are scarce.** Real degradations aren't cleanly labelled. → DSO2 is **unsupervised** (train on normal, score deviation) rather than supervised.
3. **Leakage risk in RAT labels.** Some features mechanically define the underservice label. → DSO3 explicitly **drops formula inputs/proxies** and validates with cross-validation (lesson already applied in the training notebook).
4. **Months are partially real, partially simulated.** Real anchors (Feb/Mar BSS; Mar/Apr OSS) with a documented bootstrap recipe filling gaps — the simulator is **subordinate to** real data, never a replacement.

### 5.3 Data adequacy verdict

| DSO | Needs | Available? |
|---|---|---|
| DSO1 | Labelled per-subscriber experience signal at scale | ✅ 2.47M subscribers, 5 months |
| DSO2 | Large pool of "normal" OSS behaviour | ✅ 18.8M real cell KPIs |
| DSO3 | Subscriber features + clean underservice label | ✅ leakage-controlled feature set |
| DSO4 | Time-aligned OSS & CEM series per area | ✅ multi-month aligned series |

**Verdict: the data supports all four Data Science Objectives.** Volume, temporal coverage, and structure are sufficient; the known limitations (no IMSI↔cell link, scarce anomaly labels, leakage risk) are *already reflected in the objective design* above — which is exactly what CRISP-DM Data Understanding is supposed to surface before modelling begins.

## 6. Summary — the one-slide argument

1. **The pain is real and specific:** the network degrades silently until a complaint reaches Care.
2. **Four business objectives** attack that pain: kill the blind spot (BO1), converge O+B (BO2), automate operations (BO3), protect experience (BO4).
3. **Four data-science objectives** make each business goal *measurable* and are each backed by a **real deployed model** — CEM scoring (DSO1), anomaly VAE (DSO2), RAT underservice (DSO3), Granger lead-time (DSO4).
4. **The traceability matrix proves it holds together:** no business promise is unbacked, no model is unjustified.
5. **The data understanding confirms feasibility:** real TT data at scale supports every objective, and its known limitations are already designed into the approach.

> *This notebook closes CRISP-DM Phases 1–2. The remaining phases — Data Preparation (`01`), Modelling (`02`/`03`/`04`/`10`), Evaluation, and Deployment — each execute against the objectives locked here.*